In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'RDpcUmVwb3NpdG9yaWVzXEFENjk4LWdlbmVyYXRpdmUtYWktZm9yLUJB'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"C:\\Program Files\\Python312\\Lib\\importlib\\_bootstrap.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\importlib\\_bootstrap_external.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\zipimport.py": 1744131414.0, "C:\\Program Files\\Python312\\Lib\\codecs.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\aliases.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\__init__.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\utf_8.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\encodings\\cp1252.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\abc.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\io.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\stat.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\_collections_abc.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\genericpath.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\ntpath.py": 1744131412.0, "C:\\Program Files\\Python312\\Lib\\o

In [2]:
# | echo: false
# | output: html
# |

import pandas as pd
from IPython.display import display, Markdown, HTML

sections = pd.read_excel("data/AD698-Schedule.xlsx", sheet_name="Sections")

def restrict_width(column):
    return [("max-width", "200px"), ("overflow", "hidden"), ("text-overflow", "ellipsis")]


styled_table = (
    sections.style
    .hide(axis="index")  # Hide the index
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "left")]},
        {"selector": "tbody td:nth-child(2)", "props": restrict_width(None)},
        # nth-child(4) targets the 4th column
        {"selector": "tbody td:nth-child(3)", "props": restrict_width(None)}
    ])
)

# Render and display the styled table inline in Quarto

display(HTML(styled_table.to_html()))

Section,Title,Instructor,Day,Time
A1,Generative Artificial Intelligence for Business Analytics,Nakul Padalkar,Mon,6:00PM
O,Generative Artificial Intelligence for Business Analytics,Nakul Padalkar,Tue,7:00PM


In [3]:
#| label: build-course-schedule
#| echo: false
#| warning: false
#| output: html

import pandas as pd
from IPython.display import display, HTML
from schedules.semester_planner import build_section_schedules, get_lecture_catalog

# =================================================
# USER INPUT (only edit this section)
# =================================================
semester = "Summer"
year = 2026
modalities = ["oncampus", "o1", "o2"]

start_dates_by_semester = {
    "Spring": {
        "oncampus": "2026-01-26",
        "o2": "2026-01-29",
    },
    "Summer": {
        "oncampus": "2026-05-18",
        "o1": "2026-05-19",
        "o2": "2026-05-18",
    },
    "Fall": {
        "oncampus": "2026-09-14",
        "o1": "2026-09-17",
    },
}

section_plan = build_section_schedules(
    semester=semester,
    year=year,
    modalities=modalities,
    start_dates_by_semester=start_dates_by_semester,
)

course_description = pd.read_excel(
    "data/AD698-Schedule.xlsx",
    sheet_name="Course Details"
).drop(columns=["Modules", "Module Name"], errors="ignore")

course_description = get_lecture_catalog(course_description)
max_lectures = max(plan.lecture_count for plan in section_plan.values())
course_description = course_description.head(max_lectures).reset_index(drop=True)
course_description = course_description[["Lecture", "Title", "Description"]]

oncampus_sections = [
    s for s, p in section_plan.items() if p.modality == "oncampus"
]
online_sections = [
    s for s, p in section_plan.items() if p.modality != "oncampus"
]

for s in oncampus_sections + online_sections:
    dates = section_plan[s].lecture_dates
    course_description[s] = dates + [pd.NaT] * (max_lectures - len(dates))

def fmt(d):
    return "-" if pd.isna(d) else d.strftime("%d-%b")

for col in oncampus_sections + online_sections:
    course_description[col] = course_description[col].map(fmt)

# -------------------------------------------------
# Presentation
# -------------------------------------------------
styled_table = (
    course_description
    .style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [
            ("text-align", "left"),
            ("vertical-align", "top"),
            ("max-width", "500px"),
            ("white-space", "normal")
        ]},
    ])
)

display(HTML(styled_table.to_html()))

Lecture,Title,Description,A1,O1,O2
L1.1,Course Orientation and Reproducible GenAI Setup,Course overview; GenAI system framing; AI disclosure files and accountability.,19-May,19-May,18-May
L1.2,Language Probability and Generative Systems,Why language is probabilistic; prediction as generation; uncertainty in text; NLP as the foundation of GenAI.,21-May,21-May,20-May
L2.1,Mathematical Foundations Through Language Modeling,Vectors as meaning; probability and conditional likelihood; loss functions; entropy and perplexity introduced through language modeling.,26-May,26-May,25-May
L2.2,From Words to Structured Text,Foundations of text representation; tokenization and normalization; n-grams; static versus contextual representations; document structure awareness.,28-May,28-May,27-May
L3.1,Prompting as System Design,Prompt engineering as probabilistic control; zero and few-shot learning; in-context learning; common failure modes.,02-Jun,02-Jun,01-Jun
L3.2,Tokenization Embeddings and Semantic Geometry,Subword tokenization; embedding spaces; cosine similarity; semantic neighborhoods; semantic drift.,04-Jun,04-Jun,03-Jun
L4.1,Transformers Attention and Context,Self-attention; positional encoding; scaling intuition; why transformers replaced RNNs and LSTMs.,09-Jun,09-Jun,08-Jun
L4.2,Training Paradigms and Fine-Tuning Pipelines,Pretraining versus fine-tuning; instruction tuning; dataset formatting; overfitting and data leakage risks.,11-Jun,11-Jun,10-Jun
L5.1,Retrieval Augmented Generation Concepts,Chunking and indexing; embedding-based retrieval; grounding strategies; hallucination risks.,16-Jun,16-Jun,15-Jun
L5.2,Designing Retrieval-Augmented Pipelines,End-to-end retrieval-augmented workflows; query design; citation tracing; system limitations.,18-Jun,18-Jun,17-Jun


In [4]:
# | echo: false
# | output: html
# | eval: true

import pandas as pd
from IPython.display import display, Markdown, HTML

deliverables = pd.read_excel("data/AD698-Schedule.xlsx", sheet_name="Grade")

deliverables = deliverables[[
    "Class Activity", "Count", "Points", "Max Points"]]

numeric_cols = ["Count", "Points", "Max Points"]

# 1) Coerce strings/objects to numbers, invalids → NaN
deliverables[numeric_cols] = deliverables[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

# 2) Use pandas' nullable integer dtype (allows NaN)
deliverables[numeric_cols] = deliverables[numeric_cols].astype("Int64")

# 3) Style: show "-" for NaN without changing the data
styled = (
    deliverables.style
    .hide(axis="index")
    .format(na_rep="-")  # <- this prints "-" wherever value is NA
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "left")]},
        {"selector": "tbody td:nth-child(2)", "props": [(
            "max-width", "200px"), ("overflow", "hidden"), ("text-overflow", "ellipsis")]},
        {"selector": "tbody td:nth-child(3)", "props": [(
            "max-width", "200px"), ("overflow", "hidden"), ("text-overflow", "ellipsis")]}
    ])
)

display(HTML(styled.to_html()))

Class Activity,Count,Points,Max Points
AWS Academy Generative AI Foundations,1,100,100
Group Paper Presentation (Once per Group)*,1,50,50
Assignment,4,50,200
Managerial report with Application Demo,1,70,70
Corpus Familiarization & Chatbot Scope,1,-,-
Text Structuring & Retrieval Units,1,-,-
Embeddings & Retrieval Baseline,1,-,-
Grounded Generation & Guardrails,1,-,-
Group Project Presentation,1,40,40
Group Feedback,1,40,40
